# OSCARS XAS Demonstrator — ESRF via the ELN + BESSY

This is the **NOMAD-native** variant of `1_full_pipeline.ipynb`. Here the ESRF
ID21 data was already discovered, downloaded, and **auto-converted to NXxas
inside NOMAD** by the `nomad-semantic-web-service` ELN:

1. Create a **Dataset search request** ELN entry in this upload. Its defaults
   already target the demonstrator source — synchrotron **ESRF**, technique
   **XAS**, instrument **ID21**, 2021–2022, real ICAT+ on.
2. For a matched dataset, tick **Download Files** (`trigger_download`). The `.h5`
   is downloaded into the upload and, with **auto-convert to NXxas** on (the
   default), converted to an `NXxas` `.nxs` that becomes its own entry.

So the ESRF `.nxs` already live in this upload. This notebook picks them up,
adds **BESSY** via the NOMAD search API, runs the same ewoks/est EXAFS workflow,
and writes results back — no ESRF download/convert code here.

## Setup

Configuration is environment-driven with NORTH-friendly defaults, so this
notebook runs unchanged locally and in NORTH Jupyter on `oasis-b`. Downloaded
raw files and results are written **into this upload folder** (`downloads/`,
`results/`), so NOMAD parses them into their own entries — no separate upload
step.

The ewoks graph runs headless, so `QT_QPA_PLATFORM=offscreen` is set before any
Orange import.

In [ ]:
import os, sys, subprocess
from datetime import date
from pathlib import Path

os.environ["QT_QPA_PLATFORM"] = "offscreen"   # ewoksorange pulls in Orange GUI imports

PYNX        = os.environ.get("OSCARS_PYNX", "pynx")   # pynxtools-xas CLI
NOMAD_TOKEN = os.environ.get("NOMAD_TOKEN")           # None => anonymous
BESSY_API   = os.environ.get("NOMAD_BESSY_API")       # None => oscars_demo default (staging)
BESSY_UPLOAD = os.environ.get("OSCARS_BESSY_UPLOAD", "zMg1PGypQRa4yA05cyM8Pw")

# In a dev tree, point at the semantic-web-service source instead of installing.
_sws_src = os.environ.get("NOMAD_SWS_SRC")
if _sws_src and _sws_src not in sys.path:
    sys.path.insert(0, _sws_src)


def _ensure(mods, pip_args):
    "pip-install pip_args only if any of mods is missing (skipped in a prepared kernel)."
    missing = [m for m in mods if __import__("importlib").util.find_spec(m) is None]
    if missing:
        print("installing:", " ".join(pip_args))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pip_args], check=True)


_ensure(["ewoks", "ewoksorange", "est"], ["ewoks", "ewoksorange", "est", "PyMca5", "PyQt5"])
_ensure(["requests", "yaml"], ["requests", "pyyaml"])
_ensure(["nomad_semantic_web_service"], ["nomad-semantic-web-service"])
_ensure(["jupyterlab_h5web"], ["jupyterlab-h5web"])

sys.path.insert(0, str(Path.cwd()))   # so `import oscars_demo` works in NORTH
import oscars_demo as od

DOWNLOADS = Path("downloads"); DOWNLOADS.mkdir(exist_ok=True)
RESULTS   = Path("results");   RESULTS.mkdir(exist_ok=True)
print("ready.")

## Pick up the ESRF NXxas files the ELN already produced

The ELN writes each converted file to `dataset-<id>/<stem>.nxs` under this
upload. We glob for NXxas `.nxs` already present (excluding anything this
notebook wrote under `downloads/`).

In [ ]:
esrf_nxs = sorted(
    p for p in Path.cwd().rglob("*.nxs")
    if DOWNLOADS not in p.parents and RESULTS not in p.parents
)
print(f"found {len(esrf_nxs)} ESRF NXxas file(s) from the ELN:")
for p in esrf_nxs:
    print("   ", p.relative_to(Path.cwd()))
if not esrf_nxs:
    print("\n(none yet — run the Dataset search request ELN + Download Files first)")

### View one downloaded file with H5Web

H5Web renders any HDF5/NeXus file inline. Point it at one of the converted
NXxas files to browse the spectra.

In [ ]:
from jupyterlab_h5web import H5Web

H5Web(esrf_nxs[0] if esrf_nxs else print("no file to show"))

## BESSY II — discover NXxas via the NOMAD search API

BESSY data already lives in a NOMAD oasis as **NXxas `.nxs` entries**. We search
*as if we didn't know where it was*, by the pynxtools NeXus **definition**
(`definition == NXxas`) — the facility-agnostic discovery contribution — then
download the raw `.nxs` (already NXxas, no conversion needed).

In [ ]:
bessy_hits = od.search_nxxas_entries(
    nomad_api=BESSY_API or od.NOMAD_STAGING,
    definition="NXxas",
    upload_id=BESSY_UPLOAD,
    token=NOMAD_TOKEN,
)
print(f"BESSY: {len(bessy_hits)} NXxas entries")
bessy_nxs = []
for hit in bessy_hits:
    # Tolerate a per-entry download hiccup (e.g. a transient rate limit) so one
    # bad entry doesn't abort discovery of the rest.
    try:
        got = od.download_nxs_entry(
            hit["entry_id"], DOWNLOADS / "bessy", nomad_api=hit["nomad_api"], token=NOMAD_TOKEN
        )
        bessy_nxs.extend(got)
    except Exception as exc:
        print(f"   [skip] {hit['entry_id']}: {type(exc).__name__}: {exc}")
print("BESSY .nxs:", [p.name for p in bessy_nxs])

## Common tail — ewoks/est EXAFS workflow + result entries

Every discovered `.nxs` runs through the **same** workflow (Input → Normalization
→ EXAFS → k-weight → Fourier transform → Output), driven headlessly by
`run_pymca_demo.py`. Base-`NXxas` files (BESSY foils and the ESRF ID21
fluorescence files alike) carry the signal at `entry/intensity`, so we pass
`signal="intensity"`; `run_ewoks_exafs` also strips non-physical "a.u." unit
tags that est/PyMca's pint would misread. One `.archive.yaml` per result is
written into this upload, so each result becomes its own NOMAD entry.

In [ ]:
# Every converted/discovered NXxas file is now an entry in this upload. Run the
# EXAFS workflow on a representative subset (a per-facility cap) to keep the demo
# quick; raise OSCARS_EWOKS_MAX to process more.
EWOKS_MAX = int(os.environ.get("OSCARS_EWOKS_MAX", "2"))
jobs = (
    [("ESRF@ID21", p) for p in esrf_nxs[:EWOKS_MAX]]
    + [("BESSY", p) for p in bessy_nxs[:EWOKS_MAX]]
)
print("ewoks jobs:", [(f, Path(p).name) for f, p in jobs])

In [ ]:
results = []
for facility, nxs in jobs:
    out_h5 = RESULTS / f"{Path(nxs).stem}_result.h5"
    try:
        res = od.run_ewoks_exafs(nxs, out_h5, signal="intensity", energy_unit="electron_volt")
        results.append((facility, Path(nxs).stem, nxs, res["result_file"]))
        print(f"[OK]   {facility:<12} {Path(nxs).stem}: {Path(res['result_file']).name}")
    except Exception as exc:
        print(f"[FAIL] {facility:<12} {Path(nxs).stem}: {type(exc).__name__}: {exc}")

for facility, sample, nxs, result_file in results:
    od.write_result_entry(
        RESULTS / f"{Path(nxs).stem}.archive.yaml",
        source_facility=facility, source_id=sample,
        nxs_file=str(nxs), result_file=str(result_file), sample_name=sample,
    )
print(f"\nwrote {len(results)} result entr{'y' if len(results)==1 else 'ies'} into {RESULTS}/")

## View a result with H5Web

H5Web renders any HDF5/NeXus file inline. Point it at one of the processed
result files (or at a converted NXxas input) to browse the spectra.

In [ ]:
from jupyterlab_h5web import H5Web

H5Web(str(results[0][3])) if results else print("no results to show")